# Oracles and Smart Contracts — The Notary Learns to Fill In a Form

We have been building one public proof-of-stake chain. Notebook 2 gave it validators and a stake-weighted proposer. Notebook 5 gave it a waiting room: each node has its own mempool, gossip is imperfect, and a proposer can only include what it has heard. The chain is still a **notary**. It stamps what it is handed. It does not wander outside for the price of ETH.

A **smart contract** is the form the notary agrees to fill in: if a transaction names this address and this method, run these rules. Anyone can gossip a call. A PoS proposer includes it if they have heard it.

**Question:** if a contract can only see its own storage and the arguments you pass it, how does a lending protocol know what collateral is worth?

**Scope:** a deterministic teaching model, not a recipe for shipping or attacking a real protocol. No bytecode, no gas schedule, no lawyers. This is the public PoS chain from notebooks 2 and 5, now running code.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Recap

Public proof-of-stake from notebook 2, plus the waiting room from notebook 5.

| Notebook | What it established |
| --- | --- |
| 2 | Validators, a stake-weighted proposer, slashing as hostage. |
| 5 | Each node has its own mempool. `broadcast`, then `include`. The chain is a notary, not a psychic. |

We **import** those classes. We do not reinvent `Block`, `Blockchain`, `Validator`, `Transaction`, or `Network`. We give them something to run.

The new classes below — `SmartContract`, `AMMPool`, `LendingProtocol`, `PriceFeed`, `MedianOracle` — are defined inline so the lesson stays in one place. Notebook 7 imports the shared copy from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). Same handoff as mempool after notebook 5.


## 1. What a smart contract actually is

The chain does not understand "loan" or "swap". It understands: this address, this method, these arguments. On Ethereum the rules would be bytecode. Here they are Python, because we are trying to remember an idea, not compile Solidity in a cafe.

`SmartContract` is deliberately thin: an `address`, and `call(method, **kwargs)` which runs a public method by name. Private names (`_...`) are not part of the surface. Three contracts will inherit it — a swap pool, a lending protocol, and a price feed — because those are the forms that will live at addresses on the chain we already built.

> Pause and predict: if the notary fills in the form correctly, does that mean the form was a good idea?


In [3]:
import random
import statistics
from dataclasses import dataclass

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator


class SmartContract:
    """Code that lives at an address and runs when something calls it.

    The chain does not understand loans or swaps. It understands: this
    address, this method, these arguments. The methods below are the rules.
    """

    def __init__(self, address: str) -> None:
        if not address:
            raise ValueError("Contract address must be non-empty.")
        self.address = address

    def call(self, method: str, **kwargs):
        """Invoke a public method by name."""
        if method.startswith("_") or method == "call":
            raise AttributeError(
                f"{self.address} has no public method {method!r}."
            )
        func = getattr(self, method, None)
        if not callable(func):
            raise AttributeError(  # noqa: TRY004
                f"{self.address} has no public method {method!r}."
            )
        return func(**kwargs)


def submit_call(network, chain, origin, tx_id, contract, method, **kwargs):
    """Gossip a contract call, include it, then run it.

    Inclusion is the notary stamp. ``call`` is the form being filled in.
    A proposer who never heard the gossip cannot stamp it — notebook 5
    still applies.
    """
    arg_preview = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
    description = f"{contract.address}.{method}({arg_preview})"
    network.broadcast(Transaction(tx_id, description), origin=origin)
    tx, block = network.include(origin, tx_id, chain)
    result = contract.call(method, **kwargs)
    return result, tx, block


## 2. The swap contract: a puddle with a price

A constant-product AMM keeps `x * y = k`. Here `x` is ETH reserve and `y` is USD reserve, so the displayed spot price is `USD reserve / ETH reserve`. That number is not a journalist. It is the current ratio of two piles of tokens. We omit fees, slippage limits, and external arbitrage so the splash is visible.

This is a **smart contract**: it has an address, and a swap is a method call.

We start with 50 ETH and $100,000 ($2,000/ETH) and sell 1 ETH and 40 ETH into **separate fresh pools**. A pebble and a boulder, same puddle, different splash.

> Pause and predict: which sale moves the displayed price more, and does `x * y` remain essentially unchanged?


In [5]:
class AMMPool(SmartContract):
    """A constant-product (x*y=k) two-asset market maker."""

    def __init__(
        self, address: str, eth_reserve: float, usd_reserve: float
    ) -> None:
        super().__init__(address)
        if eth_reserve <= 0 or usd_reserve <= 0:
            raise ValueError("AMM reserves must be positive.")
        self.eth_reserve = eth_reserve
        self.usd_reserve = usd_reserve

    @property
    def spot_price(self) -> float:
        """Current displayed price: USD reserve per unit of ETH reserve."""
        return self.usd_reserve / self.eth_reserve

    @property
    def constant_product(self) -> float:
        """The invariant ``x * y`` a swap should preserve (no fees here)."""
        return self.eth_reserve * self.usd_reserve

    def swap_eth_for_usd(self, eth_in: float) -> float:
        """Sell ETH into the pool, moving both reserves and the spot price."""
        if eth_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_eth_reserve = self.eth_reserve + eth_in
        new_usd_reserve = k / new_eth_reserve
        usd_out = self.usd_reserve - new_usd_reserve
        self.eth_reserve, self.usd_reserve = new_eth_reserve, new_usd_reserve
        return usd_out

    def swap_usd_for_eth(self, usd_in: float) -> float:
        """Sell USD into the pool, moving both reserves and the spot price."""
        if usd_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_usd_reserve = self.usd_reserve + usd_in
        new_eth_reserve = k / new_usd_reserve
        eth_out = self.eth_reserve - new_eth_reserve
        self.usd_reserve, self.eth_reserve = new_usd_reserve, new_eth_reserve
        return eth_out


In [6]:
initial_pool = AMMPool("amm-initial", 50.0, 100_000.0)
small_trade_pool = AMMPool("amm-pebble", 50.0, 100_000.0)
large_trade_pool = AMMPool("amm-boulder", 50.0, 100_000.0)

small_usd_out = small_trade_pool.call("swap_eth_for_usd", eth_in=1.0)
large_usd_out = large_trade_pool.call("swap_eth_for_usd", eth_in=40.0)
small_impact = (small_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
large_impact = (large_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
constant_product_preserved = abs(
    large_trade_pool.constant_product - initial_pool.constant_product
) < 1e-6

print(
    f"Initial pool: {initial_pool.eth_reserve:.2f} ETH and "
    f"${initial_pool.usd_reserve:,.2f}; spot price ${initial_pool.spot_price:,.2f}/ETH"
)
print(
    f"Small 1 ETH sale: receives ${small_usd_out:,.2f}; "
    f"price ${small_trade_pool.spot_price:,.2f}/ETH ({small_impact:.2f}%)"
)
print(
    f"Large 40 ETH sale: receives ${large_usd_out:,.2f}; "
    f"price ${large_trade_pool.spot_price:,.2f}/ETH ({large_impact:.2f}%)"
)
print(f"x * y preserved within floating-point tolerance: {constant_product_preserved}")


Initial pool: 50.00 ETH and $100,000.00; spot price $2,000.00/ETH
Small 1 ETH sale: receives $1,960.78; price $1,922.34/ETH (-3.88%)
Large 40 ETH sale: receives $44,444.44; price $617.28/ETH (-69.14%)
x * y preserved within floating-point tolerance: True


**Read the result:** the 40 ETH sale moves the displayed price by about 69%, far more than the 1 ETH sale. Nothing says ETH became 69% cheaper everywhere; this thin pool observed its own reserves. The AMM is an honest reporter of a local puddle. The constant product stays stable apart from ordinary floating-point rounding.

Remember **$617/ETH** and "a boulder in a puddle". Notebook 7 is where someone actually shoves this puddle.


## 3. The lending contract: a loan that asks the puddle

The protocol liquidates whenever collateral value / debt falls below 1.5. Its mistake is intentionally narrow: it reads `pool.spot_price` directly. That is an **oracle** choice, even though nobody named a courier. A contract cannot squint at an exchange. Whatever it treats as the price *is* its oracle.

Meet the loan we will keep bullying: **10 ETH collateral, $12,000 debt**. At a sensible ~$2,000/ETH the collateral ratio is 1.67.

Real protocols also wrestle with bonuses, partial liquidations, fees, and token transfers. We omit those so the bad price source is the only moving part.

> Pause and predict: at $2,000/ETH, is this position above or below a 1.5 threshold?


In [9]:
@dataclass(frozen=True)
class Loan:
    """A borrower's position: collateral posted against debt owed."""

    collateral_eth: float
    debt_usd: float

    def __post_init__(self) -> None:
        if self.collateral_eth <= 0 or self.debt_usd <= 0:
            raise ValueError("Loan collateral and debt must be positive.")


class LoanNotFoundError(Exception):
    """Raised when a borrower has no open loan."""


class PositionNotLiquidatableError(Exception):
    """Raised when liquidation is attempted on a still-healthy position."""


class LendingProtocol(SmartContract):
    """A toy lending protocol that liquidates undercollateralised loans.

    Its one deliberate flaw: by default it prices collateral from
    ``pool.spot_price`` unless a ``price_source`` is supplied instead.
    """

    def __init__(
        self,
        address: str,
        pool: AMMPool,
        liquidation_ratio: float = 1.5,
        price_source=None,
    ) -> None:
        super().__init__(address)
        if liquidation_ratio <= 0:
            raise ValueError("Liquidation ratio must be positive.")
        self.pool = pool
        self.liquidation_ratio = liquidation_ratio
        self.price_source = price_source
        self.loans: dict[str, Loan] = {}

    def open_loan(
        self, borrower: str, collateral_eth: float, debt_usd: float
    ) -> Loan:
        """Open a loan for ``borrower`` and return it."""
        return self.add_loan(borrower, Loan(collateral_eth, debt_usd))

    def add_loan(self, borrower: str, loan: Loan) -> Loan:
        """Register an open loan for a borrower."""
        if not borrower:
            raise ValueError("Borrower name must be non-empty.")
        self.loans[borrower] = loan
        return loan

    def collateral_ratio(self, loan: Loan) -> float:
        """Return collateral value divided by debt, using the configured price source."""
        price = (
            self.price_source.price
            if self.price_source is not None
            else self.pool.spot_price
        )
        return loan.collateral_eth * price / loan.debt_usd

    def liquidate(self, borrower: str) -> Loan:
        """Seize and remove a borrower's loan if it is unhealthy."""
        if borrower not in self.loans:
            raise LoanNotFoundError(f"No loan for {borrower}.")
        loan = self.loans[borrower]
        if self.collateral_ratio(loan) >= self.liquidation_ratio:
            raise PositionNotLiquidatableError("Position is still healthy.")
        return self.loans.pop(borrower)


In [10]:
baseline_pool = AMMPool("amm-baseline", 50.0, 100_000.0)
baseline_protocol = LendingProtocol("lending-baseline", baseline_pool)
victim_loan = baseline_protocol.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)
baseline_ratio = baseline_protocol.collateral_ratio(victim_loan)

print(f"At ${baseline_pool.spot_price:,.2f}/ETH, victim collateral ratio: {baseline_ratio:.2f}")
print("Liquidation threshold: 1.50; verdict: HEALTHY")


At $2,000.00/ETH, victim collateral ratio: 1.67
Liquidation threshold: 1.50; verdict: HEALTHY


**Read the result:** $20,000 of collateral / $12,000 debt is 1.67, so the position is healthy before anyone touches the pool. A later liquidation would not be the victim changing their loan. It would be the protocol trusting a temporary photograph of the puddle.


## 4. Put it on the chain we already built

Same `Validator`, `Blockchain`, `Network`, `Transaction`, and `Block` as notebooks 2 and 5. We instantiate an AMM at `amm.eth` and a lending protocol at `lending.eth`, then gossip two calls: Bob opens the 10 ETH / $12,000 loan, Alice swaps 1 ETH.

`submit_call` stamps the payload onto a `Block`, then runs the method. A node that missed the gossip still cannot include it.

A contract is **not** copied into every block. It lives at an address. The block records that someone called that address. After Alice's swap we will open the `Block` objects and ask: where is `amm.eth` — inside Block #4, or still at its address?

> Pause and predict: after Alice's 1 ETH swap, is Bob still above 1.5? And did `lending.eth` have to wait for another transaction before it could read `amm.eth`?


In [13]:
nodes = ["Node A", "Alice-Node", "Bob-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("Alice-Node", 80),
    Validator("Bob-Node", 70),
    Validator("Farid-Node", 50),
]
network = Network(nodes, random.Random(7))
chain = Blockchain(validators)

amm = AMMPool("amm.eth", 50.0, 100_000.0)
lending = LendingProtocol("lending.eth", amm)

network.broadcast(Transaction("tx-deploy-amm", "Deploy AMMPool at amm.eth"), origin="Node A")
network.broadcast(Transaction("tx-deploy-lending", "Deploy LendingProtocol at lending.eth"), origin="Node A")
_, deploy_amm_block = network.include("Node A", "tx-deploy-amm", chain)
_, deploy_lending_block = network.include("Node A", "tx-deploy-lending", chain)

bob_loan, bob_tx, bob_block = submit_call(
    network,
    chain,
    "Bob-Node",
    "tx-open-loan",
    lending,
    "open_loan",
    borrower="Bob",
    collateral_eth=10.0,
    debt_usd=12_000.0,
)
alice_usd, alice_tx, alice_block = submit_call(
    network,
    chain,
    "Alice-Node",
    "tx-swap-1eth",
    amm,
    "swap_eth_for_usd",
    eth_in=1.0,
)
bob_ratio_after_pebble = lending.collateral_ratio(lending.loans["Bob"])
valid, message = chain.is_valid()

print(f"Deployed {amm.address} in Block #{deploy_amm_block.index}")
print(f"Deployed {lending.address} in Block #{deploy_lending_block.index}")
print(f"{bob_block.proposer} included {bob_tx.tx_id} in Block #{bob_block.index}")
print(f"{alice_block.proposer} included {alice_tx.tx_id} in Block #{alice_block.index}")
print(
    f"After Alice's 1 ETH swap: pool spot ${amm.spot_price:,.2f}/ETH; "
    f"Bob's ratio {bob_ratio_after_pebble:.2f}"
)
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(chain.chain) - 1} inclusions)")


Deployed amm.eth in Block #1
Deployed lending.eth in Block #2
Bob-Node included tx-open-loan in Block #3
Alice-Node included tx-swap-1eth in Block #4
After Alice's 1 ETH swap: pool spot $1,922.34/ETH; Bob's ratio 1.60
Chain valid? True -- Chain is valid.
Canonical length: 5 (genesis + 4 inclusions)


### Open the `Block` objects

`include` already returned `Block` instances — the same class as notebook 2. We just have not looked inside them yet. Walk the chain. Every payload is a call. None of the payloads *is* the contract.

In [15]:
assert all(isinstance(block, Block) for block in chain.chain)

print("=== Each Block records a call. None of them stores the contract. ===")
for block in chain.chain:
    print(block)

print(f"{amm.address} still lives at one address.")
print(f"  current reserves: {amm.eth_reserve:.2f} ETH / ${amm.usd_reserve:,.2f}")
print(f"{lending.address} still lives at one address.")
print(f"  open loans: {list(lending.loans)}")
print(
    f"Block #{alice_block.index} data is only the receipt: {alice_block.data!r}"
)
print("A later block can call these addresses. They were not copied into every block.")

print()
print("=== Contract-to-contract: no extra Block ===")
print(f"{lending.address} asks {amm.address} for spot_price.")
print(f"  {amm.address}.spot_price = ${amm.spot_price:,.2f}/ETH")
print(f"  {lending.address} computes Bob's ratio as {bob_ratio_after_pebble:.2f}")
print(
    f"That inner read did not append a Block. Chain length is still "
    f"{len(chain.chain)}."
)

=== Each Block records a call. None of them stores the contract. ===
Block #0 proposed by network
  data:          Genesis Block
  previous_hash: 0000000000000000...
  hash:          02ee86b59214716d...

Block #1 proposed by Node A
  data:          Deploy AMMPool at amm.eth
  previous_hash: 02ee86b59214716d...
  hash:          25c0ad24cb350b7e...

Block #2 proposed by Node A
  data:          Deploy LendingProtocol at lending.eth
  previous_hash: 25c0ad24cb350b7e...
  hash:          13a068784d9336a2...

Block #3 proposed by Bob-Node
  data:          lending.eth.open_loan(borrower='Bob', collateral_eth=10.0, debt_usd=12000.0)
  previous_hash: 13a068784d9336a2...
  hash:          a82bd48090b233e7...

Block #4 proposed by Alice-Node
  data:          amm.eth.swap_eth_for_usd(eth_in=1.0)
  previous_hash: a82bd48090b233e7...
  hash:          167f46eb5a83c199...

amm.eth still lives at one address.
  current reserves: 51.00 ETH / $98,039.22
lending.eth still lives at one address.
  open loans:

**Read the result:** Bob's loan and Alice's swap are ordinary notebook-5 transactions. Each one produced a `Block`. Alice's pebble moves the puddle a little (~$1,922/ETH) and Bob stays healthy at about 1.60.

The contract is **not** in every block. Block #1 recorded the deploy. Block #4 recorded Alice's swap. `amm.eth` still lives at one address; its reserves changed because the call ran, not because the block swallowed the pool. If you want the contract, you look up the address. If you want the history of calls, you walk `chain.chain`.

`lending.eth` can talk to `amm.eth`. It already did: computing Bob's ratio is a contract-to-contract read of `spot_price`. That inner call did not append a Block. External transactions get stamps. Internal calls happen while a stamp is being filled in.


## 5. Different blocks, different prices

A contract cannot check an exchange screen. Someone has to *tell* it, and the way they tell it on this chain is the same way Bob opened a loan: gossip a call, get a `Block`.

An **oracle** is the input the lending contract treats as fact. That input might be `amm.spot_price`, the latest report on a feed contract, or a median of several reports. The first one already lives on-chain, which makes it tempting, and that is the trap.

The feed is a third contract, at `feed.eth`. Each report is its own transaction, so each report sits in its own `Block`. Scroll the chain: one block says $2,005, the next says $1,200, the next says $1,995. Different blocks, different numbers. The lending rule has to choose which one to believe.

Three things a stamped report is *not*, even when it lives on a valid `Block`:

- **Authenticated.** A `source` string is a label, not a signature. Nothing here proves Exchange A sent the report.
- **Fresh.** A correct price from last Tuesday is still a wrong input today. This toy has no heartbeat.
- **Trustworthy just because it is on-chain.** Readable is not the same as "the market."

> Pause and predict: if `lending.eth` trusts only the latest block's report, what happens when that block is the $1,200 lie? What if it waits and takes the median of all three reports?


In [18]:
@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who reported this price. Nothing here authenticates them.
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


def collateral_ratio(
    collateral_eth: float, debt_usd: float, eth_price_usd: float
) -> float:
    """Return collateral value divided by debt, at a given ETH price."""
    if collateral_eth <= 0 or debt_usd <= 0 or eth_price_usd <= 0:
        raise ValueError("Collateral, debt, and price must be positive.")
    return collateral_eth * eth_price_usd / debt_usd


def should_liquidate(ratio: float, threshold: float = 1.5) -> bool:
    """Return whether a collateral ratio is below the liquidation threshold."""
    return ratio < threshold


class MedianOracle:
    """A price source that reports the median of several independent reports."""

    def __init__(self, reports: list[PriceReport]) -> None:
        if not reports:
            raise ValueError("At least one price report is required.")
        self.reports = reports

    @property
    def price(self) -> float:
        """Median reported price — resists a single outlier report."""
        return statistics.median(report.eth_price_usd for report in self.reports)


class PriceFeed(SmartContract):
    """A contract that stores price reports as they arrive.

    Each ``report`` call is meant to be stamped into its own Block.
    ``latest`` is whatever the last stamp said. A ``MedianOracle`` over
    ``reports`` is a different choice.
    """

    def __init__(self, address: str) -> None:
        super().__init__(address)
        self.reports: list[PriceReport] = []

    def report(self, source: str, eth_price_usd: float) -> PriceReport:
        """Append one report and return it."""
        if eth_price_usd <= 0:
            raise ValueError("Reported price must be positive.")
        posted = PriceReport(source, eth_price_usd)
        self.reports.append(posted)
        return posted

    @property
    def latest(self) -> PriceReport:
        """The most recently stamped report."""
        return self.reports[-1]


In [19]:
feed = PriceFeed("feed.eth")
network.broadcast(
    Transaction("tx-deploy-feed", "Deploy PriceFeed at feed.eth"), origin="Node A"
)
_, deploy_feed_block = network.include("Node A", "tx-deploy-feed", chain)
assert isinstance(deploy_feed_block, Block)

collateral_eth = 10
debt_usd = 12_000
incoming_reports = [
    ("tx-report-a", "independent-feed-a", 2_005.0),
    ("tx-report-lie", "malicious-feed", 1_200.0),
    ("tx-report-b", "independent-feed-b", 1_995.0),
]

print(f"Same loan: {collateral_eth} ETH / ${debt_usd:,} debt. Threshold 1.50.")
print(f"Deployed {feed.address} in Block #{deploy_feed_block.index}")
print()

for tx_id, source, price in incoming_reports:
    _, tx, block = submit_call(
        network,
        chain,
        "Node A",
        tx_id,
        feed,
        "report",
        source=source,
        eth_price_usd=price,
    )
    assert isinstance(block, Block)
    latest_ratio = collateral_ratio(
        collateral_eth, debt_usd, feed.latest.eth_price_usd
    )
    latest_verdict = (
        "LIQUIDATE (wrong)" if should_liquidate(latest_ratio) else "HEALTHY"
    )
    print(block)
    print(
        f"  latest on {feed.address}: ${feed.latest.eth_price_usd:,.0f}/ETH "
        f"from {feed.latest.source}"
    )
    print(
        f"  if lending trusts only Block #{block.index}: "
        f"ratio {latest_ratio:.2f} -> {latest_verdict}"
    )
    print()

median_oracle = MedianOracle(feed.reports)
median_ratio = collateral_ratio(collateral_eth, debt_usd, median_oracle.price)
print(
    f"Median of every report stored at {feed.address}: "
    f"${median_oracle.price:,.0f}/ETH"
)
print(
    f"If lending reads the median instead of the latest Block: "
    f"ratio {median_ratio:.2f} -> HEALTHY"
)
print(
    f"Chain length: {len(chain.chain)}. "
    "The three reports are still sitting on their own Blocks."
)


Same loan: 10 ETH / $12,000 debt. Threshold 1.50.
Deployed feed.eth in Block #5

Block #6 proposed by Node A
  data:          feed.eth.report(source='independent-feed-a', eth_price_usd=2005.0)
  previous_hash: 90d4b6f262ea07ff...
  hash:          188772a36710b029...

  latest on feed.eth: $2,005/ETH from independent-feed-a
  if lending trusts only Block #6: ratio 1.67 -> HEALTHY

Block #7 proposed by Node A
  data:          feed.eth.report(source='malicious-feed', eth_price_usd=1200.0)
  previous_hash: 188772a36710b029...
  hash:          d7089ce65d1d1518...

  latest on feed.eth: $1,200/ETH from malicious-feed
  if lending trusts only Block #7: ratio 1.00 -> LIQUIDATE (wrong)

Block #8 proposed by Node A
  data:          feed.eth.report(source='independent-feed-b', eth_price_usd=1995.0)
  previous_hash: d7089ce65d1d1518...
  hash:          90c9740cc5c0ef63...

  latest on feed.eth: $1,995/ETH from independent-feed-b
  if lending trusts only Block #8: ratio 1.66 -> HEALTHY

Median of e

**Read the result:** three reports, three `Block`s, three different numbers. When the $1,200 lie is the latest stamp, a lending rule that trusts only the tip liquidates a healthy loan. The next block arrives at $1,995 and the same rule flips back to healthy. Different blocks, different oracles — if "oracle" means "whatever this contract just read."

The median of all three reports stays near $1,995 and the ratio 1.66 stays healthy. Aggregation raises the cost of one liar. It does not prove the feeds were signed, or fresh, or that `lending.eth` will actually read the median instead of `amm.spot_price`.

On-chain readable is not the same as trustworthy. A puddle is not the ocean. A valid `Block` is not a journalist.

That is the lesson: contracts live at addresses, they can call each other, and the numbers they treat as facts arrive one block at a time. Notebook 7 is where Alice shoves the puddle.


## Takeaways

- **Mechanism:** a smart contract lives at an address. **Not a guarantee:** a copy of it sits inside every `Block`.
- **Mechanism:** a `Block` records that someone called that address. **Not a guarantee:** the block is the contract's current storage.
- **Mechanism:** one contract can call another while a transaction is being filled in. **Not a guarantee:** that inner read appends its own block, or that the callee was a good price source.
- **Mechanism:** `broadcast`, then `include`, then `call` — same waiting room as notebook 5. **Not a guarantee:** a missed gossip still lets you include it.
- **Mechanism:** an AMM spot price honestly reflects its own reserves. **Not a guarantee:** that local ratio is "the market."
- **Mechanism:** an oracle report is another transaction, so different blocks can carry different numbers. **Not a guarantee:** the latest stamp, or a median of stamps, is true.

Next: [7. flash_loans.ipynb](7.%20flash_loans.ipynb) — same contracts, now shoved: first with owned ETH, then with a flash loan.


## Sources

Contracts at addresses, AMMs, and oracles as *inputs a contract treats as fact* all have homes outside this notebook:

- Szabo, N. (1997). [The Idea of Smart Contracts](https://nakamotoinstitute.org/library/the-idea-of-smart-contracts/). Code that fills in an agreement when named conditions hold — a vending machine, not a lawyer.
- Buterin, V. (2014). [Ethereum White Paper](https://ethereum.org/en/whitepaper/). Accounts, contract code, and calls from one contract to another on a public chain.
- Adams, H., Zinsmeister, N., & Robinson, D. (2020). [Uniswap v2 Core](https://uniswap.org/whitepaper.pdf). Constant-product pools (`x * y = k`) and why the displayed price is just the reserve ratio.
- Ellis, S., Juels, A., & Nazarov, S. (2017). [ChainLink: A Decentralized Oracle Network](https://research.chain.link/whitepaper-v1.pdf). Why a contract cannot see an exchange screen, and why one report is a different choice from a median of reports.
